### 例 15.3：油田开发动态指标灰色预测模型  

#### 背景与目标  
针对某油田 S 油藏 1994 年 2 月 - 1995 年 2 月的开发数据（月产油量、月产水量、月注水量、地层压力），构建 **灰色 GM 系列模型**（GM(1,1)、GM(1,2)、GM(1,3)），预测 1995 年 3 月的开发指标 。  

**表 15.3 某油田 S 油藏数据**

| 序号 | 时间  | 月产油量/万吨 | 月产水量/万吨 | 月注水量/万吨 | 地层压力/MPa |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 1 | 94.02 | 7.123 | 0.796 | 13.108 | 27.475 |
| 2 | 94.03 | 7.994 | 0.832 | 12.334 | 27.473 |
| 3 | 94.04 | 8.272 | 0.917 | 12.216 | 27.490 |
| 4 | 94.05 | 7.960 | 0.976 | 12.201 | 27.500 |
| 5 | 94.06 | 7.147 | 1.075 | 12.132 | 27.510 |
| 6 | 94.07 | 7.092 | 1.121 | 11.990 | 27.542 |
| 7 | 94.08 | 6.858 | 1.281 | 11.926 | 27.536 |
| 8 | 94.09 | 5.804 | 1.350 | 10.478 | 27.550 |
| 9 | 94.10 | 6.433 | 1.410 | 9.176 | 27.567 |
| 10 | 94.11 | 6.354 | 1.432 | 11.368 | 27.584 |
| 11 | 94.12 | 6.254 | 1.507 | 12.764 | 27.600 |
| 12 | 95.01 | 5.197 | 1.559 | 11.143 | 27.602 |
| 13 | 95.02 | 5.654 | 1.611 | 10.737 | 27.630 |

### 1. 模型设计思路  
根据指标间的关联关系，为 4 个开发指标（$ x_1 $：月产油量；$ x_2 $：月产水量；$ x_3 $：月注水量；$ x_4 $：地层压力）分别设计模型：  
- $ x_1 $、$ x_3 $：独立趋势，用 **GM(1,1)**（单变量灰色模型）；  
- $ x_2 $：受 $ x_1 $ 及水驱规律控制，用 **GM(1,2)**（双变量关联模型）；  
- $ x_4 $：与 $ x_1 $、$ x_3 $ 相关，用 **GM(1,3)**（三变量关联模型） 。  


### 2. 建模核心步骤  
#### （1）状态方程构建  
系统 4 个指标的微分状态方程（反映指标变化率与自身及关联指标的关系）：  
$$
\begin{cases} 
\frac{dx_1^{(1)}(t)}{dt} = a_{11}x_1^{(1)}(t) + b_1, \\
\frac{dx_2^{(1)}(t)}{dt} = a_{21}x_1^{(1)}(t) + a_{22}x_2^{(1)}(t), \\
\frac{dx_3^{(1)}(t)}{dt} = a_{33}x_3^{(1)}(t) + b_3, \\
\frac{dx_4^{(1)}(t)}{dt} = a_{41}x_1^{(1)}(t) + a_{43}x_3^{(1)}(t) + a_{44}x_4^{(1)}(t). 
\end{cases} \tag{15.10}
$$  
其中 $ x_i^{(1)} $ 是 $ x_i $ 的累加生成序列，用于弱化原始数据随机性 。  

#### （2）矩阵方程转换  
将微分方程组 (15.10) 改写为矩阵形式（便于数值求解）：  
$$
\frac{d\boldsymbol{X}^{(1)}(t)}{dt} = \boldsymbol{A}\boldsymbol{X}^{(1)}(t) + \tilde{\boldsymbol{B}}, \tag{15.11}
$$  
式中：  
- $ \boldsymbol{X}^{(1)}(t) = [x_1^{(1)}(t), x_2^{(1)}(t), x_3^{(1)}(t), x_4^{(1)}(t)]^T $（累加指标向量）；  
- $ \boldsymbol{A} $（系数矩阵）、$ \tilde{\boldsymbol{B}} $（常数项向量）通过 **最小二乘法** 辨识原始数据得到：  
  $$
  \boldsymbol{A} = \begin{bmatrix} 
  -0.0378 & 0 & 0 & 0 \\
  0.0568 & -0.2232 & 0 & 0 \\
  0 & 0 & -0.0114 & 0 \\
  -0.6179 & 0 & 5.6950 & -2.1926 
  \end{bmatrix}, \quad 
  \tilde{\boldsymbol{B}} = \begin{bmatrix} 8.6667 \\ 0 \\ 12.4938 \\ 0 \end{bmatrix}
  $$  


#### （3）初值与数值求解  
以 1994 年 2 月数据为初值 $ \boldsymbol{X}^{(1)}(1) = [7.123, 0.796, 13.108, 27.475] $，通过 **Python 求解微分方程组**（如 `scipy.integrate.solve_ivp`），得到累加序列预测值，再经 **累减还原** 得到原始指标预测值 。  


### 3. 预测结果  
1995 年 3 月（即第 14 个时间点）的预测值：  
- 月产油量 $ x_1 \approx 5.2347 $ 万吨；  
- 月产水量 $ x_2 \approx 1.4599 $ 万吨；  
- 月注水量 $ x_3 \approx 10.7070 $ 万吨；  
- 地层压力 $ x_4 \approx 26.5756 $ MPa 。  

模型验证显示，模拟值与实际值整体符合较好（注水量预测误差相对较大，需进一步优化）。  


### 4. 关键价值  
通过灰色 GM 系列模型，有效利用小样本开发数据，实现多指标关联预测，为油田开发动态调整提供量化依据，尤其适用于数据量少、关联复杂的场景 。


In [ ]:
import numpy as np
from scipy.integrate import odeint

a = np.loadtxt("Pdata15_3.txt")
n = a.shape[0]  # 观测数据的个数
x10 = a[:, 0]  # 月产油量, (13,)
x20 = a[:, 1]  # 月产水量
x30 = a[:, 2]  # 月注水量
x40 = a[:, 3]  # 地层压力
x11 = np.cumsum(x10)  # 累加生成数列, (13,)
x21 = np.cumsum(x20)
x31 = np.cumsum(x30)
x41 = np.cumsum(x40)
z1 = (x11[:-1] + x11[1:]) / 2  # 均值生成序列, (12,)
z2 = (x21[:-1] + x21[1:]) / 2
z3 = (x31[:-1] + x31[1:]) / 2
z4 = (x41[:-1] + x41[1:]) / 2

# 求解时间响应式系数
B1 = np.c_[z1, np.ones((n-1, 1))]  # (12, 2)
u1 = np.linalg.pinv(B1) @ x10[1:]  # (2,) x10[1:](12,)
print(u1)
B2 = np.c_[z1, z2]  # (12, 2)
u2 = np.linalg.pinv(B2) @ x20[1:]  # (2,)
print(u2)
B3 = np.c_[z3, np.ones((n-1, 1))]  # (12, 2)
u3 = np.linalg.pinv(B3) @ x30[1:]  # (2,)
print(u3)
B4 = np.c_[z1, z3, z4]  # (12, 3)
u4 = np.linalg.pinv(B4) @ x40[1:]  # (3,)
print(u4)

# 定义函数得到时间响应式
def func(x, t):
    x1, x2, x3, x4 = x
    return np.array([u1[0] * x1 + u1[1],
                     u2[0] * x1 + u2[1] * x2,
                     u3[0] * x3 + u3[1],
                     u4[0] * x1 + u4[1] * x3 + u4[2] * x4])
    
t = np.arange(0, 14)  # (14,)
X0 = np.array([7.1230, 0.7960, 13.1080, 27.475])  # (4,)
s1 = odeint(func, X0, t)  # (len(t), len(X0列表))(14, 4), odeint(ordinary differential equation integration常微分方程积分)
s2 = np.diff(s1, axis=0)  # (13, 4)还原原始序列，axis=0表示第一个维度,二维数组指行
xh = np.vstack([X0, s2])  # (14, 4)
cha = a - xh[:-1, :]  # 计算残差, (13, 4)
delta = np.abs(cha / a)  # 计算相对误差 (13, 4)
maxd = delta.max(0)  # 计算每个指标的最大相对误差, 0即axis=0列最大, (4,)
pre = xh[-1, :]  # (4,)
print(f"最大相对误差:\n{maxd}")
print(f"预测值为: {pre}")

[-0.03781356  8.6667348 ]
[ 0.05681482 -0.22318458]
[-1.13858518e-02  1.24938166e+01]
[-0.61790385  5.69496608 -2.18261089]
最大相对误差:
[0.13161393 0.49228926 0.23520444 0.20570548]
预测值为: [ 5.23470673  1.45993391 10.70700942 26.57561822]


In [12]:
s1

array([[  7.123     ,   0.796     ,  13.108     ,  27.475     ],
       [ 15.36360359,   1.21841534,  25.38256009,  49.2966534 ],
       [ 23.29841851,   1.96783131,  37.51815647,  77.93954472],
       [ 30.93879182,   2.96356739,  49.51636238, 107.10713337],
       [ 38.29564954,   4.14164841,  61.37873327, 136.09033972],
       [ 45.37951222,   5.45144196,  73.10680695, 164.81011476],
       [ 52.20051006,   6.85296902,  84.70210385, 193.25850276],
       [ 58.76839733,   8.3147537 ,  96.16612716, 221.43561866],
       [ 65.09256633,   9.81210349, 107.50036308, 249.3425399 ],
       [ 71.18206086,  11.32573381, 118.70628095, 276.98050252],
       [ 77.04558913,  12.84066952, 129.78533351, 304.35080645],
       [ 82.69153617,  14.34536576, 140.73895703, 331.45480225],
       [ 88.12797591,  15.83100584, 151.56857153, 358.29389383],
       [ 93.36268264,  17.29093975, 162.27558096, 384.86951204]])